# 04 — Build the network and analyze structure

Builds the bipartite contributor-repo graph, projects it onto the repo side,
analyzes degree distribution, and compares to an Erdős-Rényi null model.

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
edges_df = pd.read_csv("../data/processed/edges.csv")
repos_df = pd.read_csv("../data/processed/repos_clean.csv")

print("Edges DataFrame:", edges_df.shape)
print("Repos DataFrame:", repos_df.shape)
print()
print("Edges head:")
print(edges_df.head())

In [ ]:
# build bipartite graph
B = nx.Graph()

repos = edges_df["repo"].unique()
contributors = edges_df["contributor"].unique()

B.add_nodes_from(repos, bipartite="repo")
B.add_nodes_from(contributors, bipartite="contributor")

for _, row in edges_df.iterrows():
    B.add_edge(row["repo"], row["contributor"], weight=row["contributions"])

print("Bipartite graph:")
print(f"  Total nodes: {B.number_of_nodes()}")
print(f"  Repo nodes: {len(repos)}")
print(f"  Contributor nodes: {len(contributors)}")
print(f"  Edges: {B.number_of_edges()}")

In [ ]:
# project onto repos: two repos connected if they share a contributor
repo_nodes = [n for n, d in B.nodes(data=True) if d["bipartite"] == "repo"]
G = nx.bipartite.weighted_projected_graph(B, repo_nodes)

print("Repo projection:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Connected components: {nx.number_connected_components(G)}")

largest_cc = max(nx.connected_components(G), key=len)
print(f"  Largest component: {len(largest_cc)} nodes")

In [ ]:
# work on the largest connected component (standard practice for network analysis)
G_main = G.subgraph(largest_cc).copy()
print(f"Working graph: {G_main.number_of_nodes()} nodes, {G_main.number_of_edges()} edges")

degrees = [d for n, d in G_main.degree()]
print(f"\nDegree stats:")
print(f"  Mean: {np.mean(degrees):.2f}")
print(f"  Median: {np.median(degrees):.0f}")
print(f"  Min: {min(degrees)}")
print(f"  Max: {max(degrees)}")

In [ ]:
# degree distribution plot on log-log scale
deg_counts = Counter(degrees)
x = sorted(deg_counts.keys())
y = [deg_counts[k] for k in x]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# linear scale
axes[0].bar(x, y, width=1.0, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count")
axes[0].set_title("Degree distribution (linear)")

# log-log scale
axes[1].loglog(x, y, "o", markersize=6, alpha=0.7)
axes[1].set_xlabel("Degree (log)")
axes[1].set_ylabel("Count (log)")
axes[1].set_title("Degree distribution (log-log)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../plots/degree_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plots/degree_distribution.png")

In [ ]:
# compare to Erdős-Rényi null model
n = G_main.number_of_nodes()
m = G_main.number_of_edges()
p = (2 * m) / (n * (n - 1))

ER = nx.erdos_renyi_graph(n, p, seed=42)

c_real = nx.average_clustering(G_main)
c_er = nx.average_clustering(ER)

print(f"Real network clustering coefficient: {c_real:.4f}")
print(f"ER null model clustering coefficient: {c_er:.4f}")
print(f"Ratio (real / ER): {c_real / c_er:.1f}x")
print()
print("ER degree mean (predicted):", round(p * (n - 1), 2))
print("Real degree mean (observed):", round(np.mean(degrees), 2))

In [ ]:
# save the graph for notebook 05
nx.write_graphml(G_main, "../data/processed/repo_network.graphml")
print(f"Saved network: {G_main.number_of_nodes()} nodes, {G_main.number_of_edges()} edges")